In [1]:
# 1. Write your custom function here!

# For this demo, we do a basic NDVI calculation, inputting and returning a pandas DataFrame.
# Note that the function must input and return a pandas DataFrame. This is a requirement.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df[["time", "X", "Y", "ndvi"]]

In [ ]:
import xee
help(xee.EarthEngineBackendEntrypoint)

In [1]:
# 2. Authenticate to Google Earth Engine, if you haven't done so already on your machine!
import ee

ee.Authenticate()

c:\anaconda3\envs\rreditmode\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


True

In [2]:
# 3. Run some Earth Engine code! This will do a basic grab of some Landsat data, with some standard filtering. 
import ee
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

# Basic cloud masking algorithm
def prep_sr_l8(image):
    # Bit 0 - Fill
    # Bit 1 - Dilated Cloud
    # Bit 2 - Cirrus
    # Bit 3 - Cloud
    # Bit 4 - Cloud Shadow
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    optical_bands = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return (image.addBands(optical_bands, None, True)
                 .addBands(thermal_bands, None, True)
                 .updateMask(qa_mask)
                 .updateMask(saturation_mask))

US_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/US")
Plumas_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/Plumas_National_forest")
WSDemo_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")
Park_Lane_Boundaries = ee.FeatureCollection("projects/robust-raster/assets/boundaries/park_lane_tahoe")
#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-12-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2018-01-01', '2018-05-31').map(prep_sr_l8).select(['SR_B4', 'SR_B5'])

c:\anaconda3\envs\rreditmode\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [ ]:
import xarray as xr

ds = xr.open_dataset(ic, engine='ee', geometry=Park_Lane_Boundaries.geometry(), crs='EPSG:3310', scale=30)

In [ ]:
print(ds)

In [ ]:
# 5. Preview the dataset and the results before doing a full run!
# This allows users to inspect the structure and content of the data to ensure it behaves as expected prior to running a full computation.
from robustraster import run

run(
    dataset=ic,
    source="ee",
    dataset_kwargs={
    'geometry': US_Boundaries,
    'crs': 'EPSG:3310',
    'scale': 30
},
    user_function=compute_ndvi,
    preview_dataset=True
)

In [ ]:
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    tune_function = False,
    max_pixels_per_tile = 40_000_000,
    dataset_config={
        'geometry': Plumas_Boundaries,
        'crs': 'EPSG:5070',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "chunks": chunks,
        "max_iterations": None,
        "output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "US_NDVI_Tiles_30m_40MIL",
        "vrt": True,
        "report": True
    },
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_config={
        "n_workers": 6,
        "threads_per_worker": 1,
        "memory_limit": "2GB",
        #"ee_max_concurrency": 30
    },
    #dask_use_docker=True,
    #dask_docker_image="adrianomdocker/rr042"
)
#[robustraster] ❌ ERROR during run(): Too Many Requests: Request was rejected because the request rate or concurrency limit was exceeded.
# EEException

In [ ]:
# Docker version
# 6. Do the full computation here!
# Click on the link to the Dask dashboard to view the computation in real time!
from robustraster import run
import pandas as pd

df = pd.DataFrame(columns=["X", "Y", "SR_B4", "SR_B5", "ndvi"])
df_list = ["ndvi"]
chunks = {"time": 1, "X": 2048, "Y": 2048}

run(
    dataset=ic,
    source="ee",
    preview_dataset = False,
    tune_function = False,
    #max_pixels_per_tile = 40_000_000,
    dataset_config={
        'geometry': Park_Lane_Boundaries,
        'crs': 'EPSG:3310',
        'scale': 30,
    },
    user_function_config={
        "user_function": compute_ndvi,
        "user_function_args": (),
        "user_function_kwargs": {},
    },
    function_tuning_config={
        "chunks": chunks,
        "max_iterations": None,
        "output_template": df_list
    },
    export_config={
        "mode": "raster",
        "file_format": "GTiff",
        "output_folder": "PL_Tuned_NDVI_Tiles_30m_40MIL",
        "vrt": True,
        "report": True
    },
        #"target_blocks_per_run": 2000},
    dask_mode="custom",
    dask_config={
        "n_workers": 4,
        "threads_per_worker": 1,
        "memory_limit": "3g",
        #"ee_max_concurrency": 30
    },
    dask_use_docker=True,
    dask_docker_image="adrianomdocker/rr042"
)

[robustraster] Found EE credentials at: C:\Users\Adriano Matos/.config/earthengine/credentials
[robustraster] Mounting EE credentials to /root/.config/earthengine/credentials
[robustraster] Mounting output: C:\Users\Adriano Matos\Documents\Python Scripts\big-raster-helper\demos\PL_Tuned_NDVI_Tiles_30m_40MIL -> /robustraster_output
Dask Scheduler started: dask-3f35b6-scheduler (ec2f6c9d7193)
Dask Worker 1 started: dask-3f35b6-worker-1 (7d7ff786c1a1) mapped to port 59704
Dask Worker 2 started: dask-3f35b6-worker-2 (84fa906a4ab0) mapped to port 59705
Dask Worker 3 started: dask-3f35b6-worker-3 (f05b0aed82c1) mapped to port 56182
Dask Worker 4 started: dask-3f35b6-worker-4 (07ae8a53541e) mapped to port 56183


c:\anaconda3\envs\rreditmode\lib\site-packages\distributed\client.py:1617: VersionMismatchWarning: Mismatched versions found

+---------+-----------------+-----------------+-----------------+
| Package | Client          | Scheduler       | Workers         |
+---------+-----------------+-----------------+-----------------+
| msgpack | 1.1.2           | 1.0.8           | 1.0.8           |
| numpy   | 2.2.6           | 2.0.1           | 2.0.1           |
| python  | 3.10.18.final.0 | 3.10.19.final.0 | 3.10.19.final.0 |
| tornado | 6.5.4           | 6.4.2           | 6.4.2           |
+---------+-----------------+-----------------+-----------------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask dashboard is available at: http://127.0.0.1:8787/status
[robustraster] Docker Dask cluster started: <Client: 'tcp://172.20.0.2:8786' processes=2 threads=2, memory=5.59 GiB>
[robustraster] Dask workers authenticated to Earth Engine.
[robustraster] AOI tiling enabled. Streaming tiles in batches...
[robustraster] Processing tile 1 of 1
LOOPDOLOOP
<xarray.Dataset> Size: 488B
Dimensions:  (time: 8, X: 3, Y: 2)
Coordinates:
  * time     (time) datetime64[ns] 64B 2018-01-01T18:39:39.007000 ... 2018-05...
  * X        (X) float64 24B -1.153e+03 -1.123e+03 -1.093e+03
  * Y        (Y) float64 16B 1.352e+05 1.353e+05
Data variables:
    SR_B4    (time, X, Y) float32 192B ...
    SR_B5    (time, X, Y) float32 192B ...
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    prod